# Experiments with our new superpowrs


In [ ]:
#| default_exp tutorials/scale

In [ ]:
import sys
import math
import numpy as np
from fastcore.basics import patch
#from HexMagic.database import GeoStorage, ChunkedTerrainGenerator, ChunkRef
from HexMagic.primitives import MapRect, MapCord, MapSize, HexPosition,PrimitiveDemo
from HexMagic.core import Terrain, TerraDemo



In [ ]:
from HexMagic.climate import TerrainPatterns, DrainageBasins, Geology, TerraDemo, Terrain, GeoBounds, ClimatePreset, TerrainFactory
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, hexBackground, HexRegion, unique_windy_edge, PrimitiveDemo, HexChunk
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.geology import Plate
from HexMagic.geology import River, Watershed



So I have spent the last few months thinking about scale as I worked on a silly map design project. My eldest kid came to me wanting an autogenerating board game map, and I had signed up for an AI class, so I thought I would combine the two. I like the idea of exploring map design and exploring maps currently. 

Like most journeys, I would start small and grow. One can, of course, jump rather far if you leverage things quickly, and there are software systems that rival intercontinental railroads. I decided not to look at those as much because they all seemed heavy, and at the end of the day, I wanted something that could easily draw a single hexagon.

In [ ]:
SVGBuilder.BUILDERHIDE = True

In [ ]:
hexStyle = StyleCSS("HexStyle",fill="yellow",stroke="blue",stroke_width=2)
canvas = SVGBuilder()
canvas.width=200 ;canvas.height=200
canvas.add_style(hexStyle)

#add our hex to the canvas
sampleHex = Hex(radius=50,center=MapCord(100,100),style=hexStyle)
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()

'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<svg  width="200" height="200" viewBox="0 0 200 200" xmlns="h'

And then could build a simple grid.

In [ ]:
sampleGrid = PrimitiveDemo().sampleGrid(fill="yellow",makeLabels = True)
sampleGrid.builder.show()

'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<svg  width="351.77" height="280.0" viewBox="0 0 351.77 280.0'

My mind was quite happy being in a nice x,y space with a simple array underneath it. I had taken enough data science classes (in part with the same teacher doing AI) that I knew things were generally better computationally if you use numpy. I need something that could give these maps depth so I created a terrain object that could use these grids.

In [ ]:
demoTerr = TerraDemo().sanFran()
demoTerr.elevations[:30]

array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 27.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  1.,  0.,  0.,  0.,  0.,
        0.,  0.,  0., 55.])

and these could be easily put into a grid. The textElevations function rounds down elevations in 100 meters increments so it is easier to put into a notebook.

In [ ]:
demoTerr.textElevations()

|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   1   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   1   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   1   0   0   0   1   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   

Having taken a bunch of ai classes and for that matter some math in college, I realized you can sample around a certain area to reduce the size of the map.

In [ ]:
demoTerr.scaled(0.25).textElevations()

|   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0 |
|   0   0   0   1   0   0   0 |
|   0   0   1   1   1   0   0 |
|   0   0   0   1   0   0   0 |
|   0   0   0   1   0   1   0 |


and things can be increased, but you will lose information if you do both

In [ ]:
backTerr = demoTerr.scaled(0.25).upsample(scale=4.0, method='bilinear')
backTerr.textElevations()

|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 |
|   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   

Those these were better if you added color as you drew them.

In [ ]:
demoTerr = TerraDemo().sanFran()
demoTerr.colorMap()
demoTerr.hexGrid.update()
textStyle = StyleCSS("textStyle",fill="white",font_size="2em",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")
demoTerr.hexGrid.builder.add_style(textStyle)
demoTerr.hexGrid.builder.add_centered_text(
        "I left my ❤️ in San Francisco", 
        y_offset=-demoTerr.hexGrid.builder.height/2 + 70,
        class_name=textStyle.name
    )

demoTerr.hexGrid.builder.show()

'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<svg  width="613.15" height="522.0" viewBox="0 0 613.15 522.0'

In [ ]:
backTerr.hexGrid.adjustRadius(demoTerr.hexGrid.radius)
backTerr.colorMap()
backTerr.hexGrid.update()
textStyle = StyleCSS("textStyle",fill="white",font_size="2em",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")

backTerr.hexGrid.builder.add_style(textStyle)
backTerr.hexGrid.builder.add_centered_text(
        "Things smooth", 
        y_offset=-backTerr.hexGrid.builder.height/2 + 40,
        class_name=textStyle.name
    )

backTerr.hexGrid.builder.show()



'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<svg  width="571.58" height="486.0" viewBox="0 0 571.58 486.0'

## A coordinate aside.
As I started to build these I realized that there must be some better math systems than I had. And I learned about cube coordinates at https://www.redblobgames.com/grids/hexagons/ 

In [ ]:
#mark the treasure
smallTer = demoTerr.scaled(0.25)
grid = smallTer.hexGrid
grid.adjustRadius(80)
builder = grid.builder
builder.layers = []
grid.hexes[grid.middle].label = "X"

#lets have a directions layer
arrowLayer = ""

#Use the realtive position class called HexPosition
for position in HexPosition.directions():
    i = grid.hexposition_to_index(position, grid.middle)
    grid.hexes[i].label = position.label
    arrowLayer += grid.arrow(grid.middle,i)

smallTer.colorMap()

grid.update()

#we can add a layer or update one in a builder using the adjust method
builder.adjust("arrows",arrowLayer)
textStyle = StyleCSS("textStyle",fill="white",font_size="9",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")
builder.add_style(textStyle)
builder.add_centered_text(
        "Would you tell me, please, which way I ought to go from here?", 
        y_offset=-builder.height/2 + 40,
        class_name=textStyle.name
    )


grid.builder.show()

'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<svg  width="900.67" height="720.0" viewBox="0 0 900.67 720.0'

I talk a great deal about cube coordinates elsewhere, and it look me the longest time to figure them out. The only way my brain could think about them was as a set of directions. These directions are always going to have a center reference point and what is nice for many algorithms is that you can move this reference point. For things like shorest path you can see how you can just add directions to the outer edges to get the next point.

In [ ]:
#mark the treasure
smallTer = demoTerr.scaled(0.25)
grid = smallTer.hexGrid
grid.adjustRadius(80)
builder = grid.builder
builder.layers = []
grid.hexes[grid.middle].label = "X"

#lets have a directions layer
arrowLayer = ""
already = set([grid.middle])
oldArrow = StyleCSS("oldArrow",stroke="gray",stroke_width=1)
newArrow = StyleCSS("newArrow",stroke="yellow",stroke_width=1)
builder.add_style(oldArrow)
builder.add_style(newArrow)

#Use the realtive position class called HexPosition
for position in HexPosition.directions():
    i = grid.hexposition_to_index(position, grid.middle)
    grid.hexes[i].label = position.label
    already.add(i)
    arrowLayer += grid.arrow(grid.middle,i,style = oldArrow)

for start in already:
    for position in HexPosition.directions():
        i = grid.hexposition_to_index(position, start)
        if i not in already:
            arrowLayer += grid.arrow(start,i,style = newArrow)
    
smallTer.colorMap()
grid.update()
grid.builder.adjust("arrows",arrowLayer)
textStyle = StyleCSS("textStyle",fill="white",font_size="9",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")
builder.add_style(textStyle)
builder.add_centered_text(
        "That depends a good deal on where you want to get to, said the Cat.", 
        y_offset=-builder.height/2 + 40,
        class_name=textStyle.name
    )

grid.builder.show()

'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<svg  width="900.67" height="720.0" viewBox="0 0 900.67 720.0'

One thing that I learned much later. was that these coordinates would be much easier to work with of instead of doing an x-y offset, that you would orient youself in the middle. This would give you a pipeline 

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐       ┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│    translate    │ ──► │    translate    │ ──► │     scale.      │  ──►  │    translate    │ ──► │       scale     │ ──► │     transform   │
│ to local center │     │    to coarse    │     │     at coarse.  │       │    at coarse    │     │  back to detail │     │ (back to orign) │
└─────────────────┘     └─────────────────┘     └─────────────────┘       └─────────────────┘     └─────────────────┘     └─────────────────┘
```

where you translate/ scale/ translate as you go through things.

In [ ]:
??HexGrid.hexposition_to_index


```python
@patch
def hexposition_to_index(self: HexGrid, hexpos: HexPosition, origin_index:int = None) -> int:
    if origin_index is None:
        origin_index = int(len(self.hexes)/2)
    """Convert HexPosition (relative to origin) back to grid index.
    Returns -1 if out of bounds."""
    # Get origin's cube coordinates
    origin_row, origin_col = self.index_to_row_col(origin_index)
    origin_q = origin_col - (origin_row - (origin_row & 1)) // 2
    origin_r = origin_row
    origin_s = -origin_q - origin_r

    # Add relative position to origin
    abs_q = hexpos.q + origin_q
    abs_r = hexpos.r + origin_r
    abs_s = hexpos.s + origin_s

    # Convert cube back to odd-r offset
    row = abs_r
    col = abs_q + (abs_r - (abs_r & 1)) // 2

    # Convert to index and check bounds
    return self.row_col_to_index(row, col)
```

**File:** `~/HexMagic/HexMagic/plot/hex.py`

In [ ]:
demoTerr = TerraDemo().sanFran()
demoTerr.colorMap()
demoTerr.hexGrid.update()
textStyle = StyleCSS("textStyle",fill="white",font_size="2em",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")
demoTerr.hexGrid.builder.add_style(textStyle)
demoTerr.hexGrid.builder.add_centered_text(
        "I don’t much care where– said Alice.", 
        y_offset=-demoTerr.hexGrid.builder.height/2 + 70,
        class_name=textStyle.name
    )

grid =  demoTerr.hexGrid
hexDim = 10
arrowLayer = ""
origin = grid.middle

for pos in [hexDim * x for x in [HexPosition.E,HexPosition.W,HexPosition.NE,HexPosition.SW]]:
    index = grid.hexposition_to_index(pos,origin_index=origin)
    arrowLayer += grid.arrow(origin,index)

grid.builder.adjust("axis",arrowLayer)


demoTerr.hexGrid.builder.show()

'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<svg  width="613.15" height="522.0" viewBox="0 0 613.15 522.0'

An important footnote is that my older style hexGrids were not center oriented. and when I created new from existing maps they use the old version which has the center as middle. For the maps that we generate they use the api called middle.

In [ ]:
??HexChunk


```python
class HexChunk:
    """A hex-shaped region of exactly N rings from center, with optional halo."""

    # Class-level cache: rings -> (spiral list, pos->idx dict)
    _spiral_cache: dict[int, tuple[list[HexPosition], dict[HexPosition, int]]] = {}


    def __init__(self,
        position: HexPosition,  # Chunk coordinate (not world)
        rings: int,
        style: StyleCSS = None,
        core_size: int = None):  # If None, all hexes are core

        self.position = position
        self.rings = rings
        self.style = style or StyleCSS("chunk_default", fill="#f0f0f0", stroke="#ccc", stroke_width=1)

        # Build/retrieve spiral mapping
        if rings not in HexChunk._spiral_cache:
            spiral = HexPosition.origin().spiral(rings)
            pos_to_idx = {pos: i for i, pos in enumerate(spiral)}
            HexChunk._spiral_cache[rings] = (spiral, pos_to_idx)

        self._spiral, self._pos_to_idx = HexChunk._spiral_cache[rings]

        # Core/halo boundary - indices < core_size are core
        self.core_size = core_size if core_size is not None else len(self._spiral)

        # Data arrays indexed by spiral order
        self.elevations = np.zeros(len(self._spiral))
        self.fields: dict[str, np.ndarray] = {}

        # Style per hex (optional override)
        self.hex_styles: list[StyleCSS | None] = [None] * len(self._spiral)

    def __len__(self):
        return len(self._spiral)

    # === Factory methods ===

    @classmethod
    def with_halo(cls, position: HexPosition, core_rings: int, halo_rings: int,
                  style: StyleCSS = None) -> 'HexChunk':
        """Create chunk with explicit core/halo split."""
        total_rings = core_rings + halo_rings
        core_size = cls.spiral_size(core_rings)
        return cls(position, total_rings, style, core_size=core_size)

    @staticmethod
    def spiral_size(rings: int) -> int:
        """Number of hexes in a spiral of N rings (including center)."""
        if rings == 0:
            return 1
        return 1 + 3 * rings * (rings + 1)

    # === Core/Halo properties ===

    @property
    def core_rings(self) -> int:
        """Number of rings in core (derived from core_size)."""
        # Inverse of spiral_size: solve 1 + 3*n*(n+1) = core_size
        # 3n² + 3n + 1 - core_size = 0
        if self.core_size <= 1:
            return 0
        n = int((np.sqrt(12 * self.core_size - 3) - 3) / 6)
        return n

    @property
    def halo_rings(self) -> int:
        """Number of rings in halo."""
        return self.rings - self.core_rings

    def is_core(self, idx: int) -> bool:
        """True if index is in core (not halo)."""
        return 0 <= idx < self.core_size

    def is_halo(self, idx: int) -> bool:
        """True if index is in halo."""
        return self.core_size <= idx < len(self)

    # === Iteration ===

    def iter_core(self) -> Iterator[int]:
        """Iterate over core indices only."""
        return iter(range(self.core_size))

    def iter_halo(self) -> Iterator[int]:
        """Iterate over halo indices only."""
        return iter(range(self.core_size, len(self)))

    def iter_all(self) -> Iterator[int]:
        """Iterate over all indices (core + halo)."""
        return iter(range(len(self)))

    def iter_core_with_pos(self) -> Iterator[ChunkLocalMap]:
        """Yield (idx, local_pos) for core hexes."""
        for idx in self.iter_core():
            yield ChunkLocalMap(idx, self._spiral[idx])

    def iter_with_world(self) -> Iterator[ChunkWorldMap]:
        """Yield (idx, local_pos, world_pos) for all hexes."""
        center = self.center_world
        for idx, local_pos in enumerate(self._spiral):
            yield ChunkWorldMap(idx, local_pos, center + local_pos)

    # === World coordinate conversion ===

    @property
    def center_world(self) -> HexPosition:
        """World position of chunk center."""
        spacing = 2 * self.core_rings  # Match world_to_chunk spacing
        return HexPosition(
            self.position.q * spacing,
            self.position.r * spacing,
            self.position.s * spacing
        )

    @property
    def key(self) -> str:
        return f"{self.position.q}_{self.position.r}_{self.position.s}"

    # === Index conversion (local) ===

    def index_to_hexposition(self, idx: int) -> HexPosition | None:
        """Convert spiral index to local HexPosition (relative to chunk center)."""
        if 0 <= idx < len(self._spiral):
            return self._spiral[idx]
        return None

    def hexposition_to_index(self, pos: HexPosition) -> int:
        """Convert local HexPosition to spiral index. Returns -1 if not in chunk."""
        return self._pos_to_idx.get(pos, -1)

    # Shorthand aliases
    i2hp = index_to_hexposition
    hp2i = hexposition_to_index

    # === World coordinate conversion ===

    def index_to_world(self, idx: int) -> HexPosition | None:
        """Convert spiral index to world HexPosition."""
        local = self.index_to_hexposition(idx)
        if local is None:
            return None
        return self.center_world + local

    def world_to_index(self, world_pos: HexPosition) -> int:
        """Convert world HexPosition to spiral index. Returns -1 if not in chunk."""
        local = world_pos - self.center_world
        return self.hexposition_to_index(local)

    # === Neighbors ===

    def neighbors_of(self, idx: int, ring: int = 1) -> list[int]:
        """Get indices of neighbors within chunk. Filters out-of-bounds."""
        local_pos = self.index_to_hexposition(idx)
        if local_pos is None:
            return []

        neighbor_positions = local_pos.ring(ring)
        return [self.hexposition_to_index(p) for p in neighbor_positions 
                if self.hexposition_to_index(p) >= 0]

    def core_neighbors_of(self, idx: int) -> list[int]:
        """Get neighbors that are in core only."""
        return [n for n in self.neighbors_of(idx) if self.is_core(n)]

    # === Edge/Border helpers ===

    def edge_indices(self, direction: int) -> list[int]:
        """Indices along chunk edge facing direction (0-5)."""
        # Edge hexes are at max distance in that direction
        edge = []
        dir_hp = HexPosition.directions()[direction]
        for idx in self.iter_core():
            pos = self._spiral[idx]
            neighbor_pos = pos + dir_hp
            # If neighbor would be outside core, this is an edge hex
            if self.hexposition_to_index(neighbor_pos) < 0 or \
               not self.is_core(self.hexposition_to_index(neighbor_pos)):
                edge.append(idx)
        return edge

    # === Data operations ===

    def trim_to_core(self) -> 'HexChunk':
        """Return new chunk with only core data (no halo)."""
        new_chunk = HexChunk(self.position, self.core_rings, self.style)
        new_chunk.elevations = self.elevations[:new_chunk.core_size].copy()
        for name, data in self.fields.items():
            new_chunk.fields[name] = data[:new_chunk.core_size].copy()
        new_chunk.hex_styles = self.hex_styles[:new_chunk.core_size].copy()
        return new_chunk

    def copy_from(self, other: 'HexChunk', world_offset: HexPosition = None):
        """Copy overlapping data from another chunk.

        Args:
            other: Source chunk
            world_offset: If None, assumes chunks share coordinate system
        """
        for other_idx, other_local, other_world in other.iter_with_world():
            my_idx = self.world_to_index(other_world)
            if my_idx >= 0:
                self.elevations[my_idx] = other.elevations[other_idx]
                for name in other.fields:
                    if name not in self.fields:
                        self.fields[name] = np.zeros(len(self))
                    self.fields[name][my_idx] = other.fields[name][other_idx]

    # === Field management ===

    def add_field(self, name: str, default: float = 0.0):
        """Add a new data field."""
        self.fields[name] = np.full(len(self), default)

    def get_field(self, name: str) -> np.ndarray | None:
        """Get field array by name."""
        return self.fields.get(name)

    # === Rendering ===

    def to_hexes(self, radius: float, center: MapCord) -> list[Hex]:
        """Generate Hex objects for rendering."""
        hexes = []
        for idx, local_pos in enumerate(self._spiral):
            pixel = local_pos.pixel(radius, center)
            style = self.hex_styles[idx] or self.style
            hexes.append(Hex(radius, pixel, style))
        return hexes

    def render_svg(self, radius: float, center: MapCord, 
                   core_only: bool = False) -> str:
        """Render chunk to SVG string."""
        svg = ""
        indices = self.iter_core() if core_only else self.iter_all()
        for idx in indices:
            local_pos = self._spiral[idx]
            pixel = local_pos.pixel(radius, center)
            style = self.hex_styles[idx] or self.style
            hex_obj = Hex(radius, pixel, style)
            svg += "\t" + hex_obj.svg() + "\n"
        return svg

    def apply_styles(self, color_levels: list[StyleCSS], 
                     elevation_delta: float = 100,
                     sea_level: StyleCSS = None):
        """Apply styles based on elevation (like Terrain.colorMap)."""
        num_levels = len(color_levels)

        for i in range(len(self)):
            elev = self.elevations[i]
            if elev <= 0 and sea_level:
                self.hex_styles[i] = sea_level
            else:
                level = int(elev / elevation_delta)
                level = min(level, num_levels - 1)
                level = max(level, 0)
                self.hex_styles[i] = color_levels[level]
```

**File:** `~/HexMagic/HexMagic/plot/chunk.py`

In [ ]:
class ChunkCover:
    """Covers a terrain with HexChunks arranged in patterns."""

    def __init__(self, terrain, rings, halo_rings=1):
        self.terrain = terrain
        self.rings = rings
        self.halo_rings = halo_rings

    def chunkAt(self, origin=None) -> HexChunk:
        """Create a chunk with halo centered at the given grid index."""
        grid = self.terrain.hexGrid
        if origin is None:
            origin = grid.middle

        chunk = HexChunk.with_halo(
            HexPosition.origin(), 
            core_rings=self.rings, 
            halo_rings=self.halo_rings
        )
        chunk.origin = origin

        for prop in self.terrain.fields.keys():
            chunk.fields[prop] = np.zeros(len(chunk))

        for i in chunk.iter_with_world():
            source = grid.hexposition_to_index(i.local_pos, origin_index=origin)
            if 0 <= source < len(self.terrain.elevations):
                chunk.elevations[i.idx] = self.terrain.elevations[source]
                for prop in self.terrain.fields.keys():
                    chunk.fields[prop][i.idx] = self.terrain.fields[prop][source]
        return chunk

    def regionAt(self, origin=None, include_halo=False) -> HexRegion:
        """Create a HexRegion covering the chunk area.
        
        Args:
            origin: Grid index for chunk center (default: grid.middle)
            include_halo: If True, include halo hexes in region
        """
        grid = self.terrain.hexGrid
        if origin is None:
            origin = grid.middle

        region = HexRegion(set(), hexGrid=grid)
        chunk = HexChunk.with_halo(
            HexPosition.origin(), 
            core_rings=self.rings, 
            halo_rings=self.halo_rings
        )
        
        indices = chunk.iter_all() if include_halo else chunk.iter_core()
        for idx in indices:
            pos = chunk.index_to_hexposition(idx)
            source = grid.hexposition_to_index(pos, origin_index=origin)
            if 0 <= source < len(grid.hexes):
                region.hexes.add(source)
        return region

    def compute_spiral_rings(self) -> int:
        """Compute number of chunk rings needed to cover the grid."""
        grid = self.terrain.hexGrid
        min_dim = min(grid.nRows, grid.nCols)
        # Each chunk covers 2*rings+1 diameter, spacing is 2*rings
        # Need enough rings to reach from center to edge
        chunk_diameter = 2 * self.rings
        return max(1, int(np.ceil(min_dim / chunk_diameter / 2)))

    def spiral_chunks(self, n_rings: int = None) -> list[HexChunk]:
        """Generate chunks arranged in a spiral pattern.
        
        Args:
            n_rings: Number of rings of chunks around center. 
                     If None, computed from grid dimensions.
        """
        grid = self.terrain.hexGrid
        
        if n_rings is None:
            n_rings = self.compute_spiral_rings()
        
        # Distance between chunk centers for edge-touching (with halo overlap)
        distance = 2 * self.rings
        
        positions = HexPosition.origin().spiral(n_rings, distance=distance)
        
        chunks = []
        for pos in positions:
            origin_idx = grid.hexposition_to_index(pos, origin_index=grid.middle)
            
            if origin_idx < 0 or origin_idx >= len(grid.hexes):
                continue
                
            chunk = self.chunkAt(origin=origin_idx)
            chunk.position = pos
            chunks.append(chunk)
        
        return chunks


In [ ]:
SVGBuilder.BUILDERHIDE = False

In [ ]:
demoTerr = TerraDemo().sanFran()
demoTerr.colorMap()
demoTerr.hexGrid.update()
textStyle = StyleCSS("textStyle",fill="white",font_size="2em",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")
demoTerr.hexGrid.builder.add_style(textStyle)
demoTerr.hexGrid.builder.add_centered_text(
        "I don’t much care where– said Alice.", 
        y_offset=-demoTerr.hexGrid.builder.height/2 + 70,
        class_name=textStyle.name
    )

grid =  demoTerr.hexGrid
hexDim = 10
arrow_style = StyleCSS("spiral_arrow", stroke="purple", stroke_width=2, fill="none")
arrowLayer = ""
chunkLayer = ""
origin = grid.middle
#cover = ChunkCover(demoTerr,rings=3)
prev_idx = None

cover = ChunkCover(demoTerr, rings=3, halo_rings=1)
chunks = cover.spiral_chunks()
# Get seaborn palette with opacity for blending
chunk_styles = StyleCSS.seaborn("husl", levels=len(chunks))
for  i, style in enumerate(chunk_styles):
    style.name = f"Chunk_{i}"
    style.opacity = 0.5  # Semi-transparent for blending
    grid.builder.add_style(style)


for i, chunk in enumerate(chunks): # auto-computes n_rings
    chunk.apply_styles(demoTerr.colorLevels, elevation_delta=demoTerr.elevationDelta)
    chunkStyle = chunk_styles[i]
    centerPos = grid.hexes[chunk.origin].center
    chunkLayer += chunk.render_svg(grid.radius, center=centerPos, core_only=False)

    for idx in chunk.iter_all():
        pos = chunk.index_to_hexposition(idx)
        grid_idx = grid.hexposition_to_index(pos, origin_index=chunk.origin)
        if 0 <= grid_idx < len(grid.hexes):
            grid.hexes[grid_idx].style = chunkStyle

     # Draw arrow from previous hex to current
    if prev_idx is not None and prev_idx >= 0:
        arrowLayer += grid.arrow(prev_idx, chunk.origin, style=arrow_style)
    
    prev_idx = chunk.origin




builder = demoTerr.hexGrid.builder
builder.layers = []
textStyle = StyleCSS("textStyle",fill="purple",font_size="2em",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")
builder.adjust("base",chunkLayer)
chunkOverlay = grid.styleLayer()
builder.adjust("chunks", chunkOverlay)
builder.adjust("arrows",arrowLayer)
builder.add_style(textStyle)
builder.add_style(arrow_style)
builder.add_centered_text(
        "Then it won't matter where you go", 
        y_offset=-demoTerr.hexGrid.builder.height/2 + 70,
        class_name=textStyle.name
    )
builder.show()
    



#demoTerr.hexGrid.builder.show()

What did I mess up?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The issue is that you're adding chunk styles to `builder` before it's defined in this cell:

```python
for  i, style in enumerate(chunk_styles):
    style.name = f"Chunk_{i}"
    style.fill_opacity = 0.5
    builder.add_style(style)  # ← builder is from a previous cell/run
```

But then later you reassign:
```python
builder = demoTerr.hexGrid.builder
builder.layers = []
```

So the styles went to whatever `builder` was before (probably from a previous cell), not to `demoTerr.hexGrid.builder`. When `grid.styleLayer()` looks up styles in `self.builder.styles`, it doesn't find them.

**Fix:** Move the `builder = demoTerr.hexGrid.builder` line to *before* the loop that adds chunk styles:

```python
builder = demoTerr.hexGrid.builder
builder.layers = []

chunk_styles = StyleCSS.seaborn("husl", levels=len(chunks))
for i, style in enumerate(chunk_styles):
    style.name = f"Chunk_{i}"
    style.fill_opacity = 0.5
    builder.add_style(style)
```

In [ ]:
??HexGrid.styleLayer


```python
@patch
def styleLayer(self: HexGrid, f=None,smooth=False):
    """Generate style layer with shared borders between regions."""
    retLayer = ""
    regions = self.styleRegions()
    borders = {}  # Shared cache across all regions
    allPaths = []

    for styleName, region in regions.items():
        style = self.builder.styles[styleName]
        allPaths.extend( region.trace_perimeter_cached(borders, f, style))

    allPaths = reversed(sorted(allPaths))
    for path in allPaths:
        if smooth:
            #path = path.smooth()
            retLayer += path.svg()
        else:
            retLayer += path.drawClosed()

    return retLayer
```

**File:** `~/HexMagic/HexMagic/plot/region.py`

can you add an overlay that shows the hexchunks by region including their halos. we would adjust the style of the hexes to something that has a unique color (from StyleCSS.seaborn) and slighly opaque then call styleLayer to create the map. we should have some nice blending on the overlaping halos

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's how to create a chunk overlay with seaborn colors and opacity for blending:

```python
demoTerr = TerraDemo().sanFran()
demoTerr.colorMap()
demoTerr.hexGrid.update()

grid = demoTerr.hexGrid
builder = grid.builder
builder.layers = []

# Get seaborn palette with opacity for blending
chunk_styles = StyleCSS.seaborn("husl", levels=12)
for style in chunk_styles:
    style.fill_opacity = 0.5  # Semi-transparent for blending
    builder.add_style(style)

# Create chunks and assign styles to grid hexes
cover = ChunkCover(demoTerr, rings=3, halo_rings=1)
chunks = cover.spiral_chunks()

for i, chunk in enumerate(chunks):
    style = chunk_styles[i % len(chunk_styles)]
    
    # Mark all hexes (core + halo) in the grid with this chunk's style
    for idx in chunk.iter_all():
        pos = chunk.index_to_hexposition(idx)
        grid_idx = grid.hexposition_to_index(pos, origin_index=chunk.origin)
        if 0 <= grid_idx < len(grid.hexes):
            grid.hexes[grid_idx].style = style

# Render base terrain
baseLayer = ""
for h in grid.hexes:
    baseLayer += h.svg()

# Use styleLayer for smooth region boundaries
chunkOverlay = grid.styleLayer()

# Assemble
builder.adjust("base", baseLayer)
builder.adjust("chunks", chunkOverlay)

textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                     filter="drop-shadow(2px 2px 2px rgba(0,0,0,.7))")
builder.add_style(textStyle)
builder.add_centered_text("Chunk regions with halo overlap", 
                          y_offset=-builder.height/2 + 50,
                          class_name=textStyle.name)
builder.show()
```

This will:
1. Use `StyleCSS.seaborn("husl", 12)` to get distinct colors
2. Set `fill_opacity=0.5` on each style for semi-transparency
3. Assign styles to grid hexes based on which chunk covers them (including halos)
4. Where halos overlap, the later chunk's style replaces earlier ones - but the opacity means you still see through to the terrain beneath

If you want true blending where overlapping halos show both colors, you'd need to render each chunk as a separate SVG group with `mix-blend-mode`. Want me to show that approach instead?

So can you rewrite ChunkCover both the regionAt (to return the proper halo) and the spiral_chunks which should be able to compute the number of rings based on the min of HexGrid.nRows or nCols

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
class ChunkCover:
    """Covers a terrain with HexChunks arranged in patterns."""

    def __init__(self, terrain, rings, halo_rings=1):
        self.terrain = terrain
        self.rings = rings
        self.halo_rings = halo_rings

    def chunkAt(self, origin=None) -> HexChunk:
        """Create a chunk with halo centered at the given grid index."""
        grid = self.terrain.hexGrid
        if origin is None:
            origin = grid.middle

        chunk = HexChunk.with_halo(
            HexPosition.origin(), 
            core_rings=self.rings, 
            halo_rings=self.halo_rings
        )
        chunk.origin = origin

        for prop in self.terrain.fields.keys():
            chunk.fields[prop] = np.zeros(len(chunk))

        for i in chunk.iter_with_world():
            source = grid.hexposition_to_index(i.local, origin_index=origin)
            if 0 <= source < len(self.terrain.elevations):
                chunk.elevations[i.idx] = self.terrain.elevations[source]
                for prop in self.terrain.fields.keys():
                    chunk.fields[prop][i.idx] = self.terrain.fields[prop][source]
        return chunk

    def regionAt(self, origin=None, include_halo=False) -> HexRegion:
        """Create a HexRegion covering the chunk area.
        
        Args:
            origin: Grid index for chunk center (default: grid.middle)
            include_halo: If True, include halo hexes in region
        """
        grid = self.terrain.hexGrid
        if origin is None:
            origin = grid.middle

        region = HexRegion(set(), hexGrid=grid)
        chunk = HexChunk.with_halo(
            HexPosition.origin(), 
            core_rings=self.rings, 
            halo_rings=self.halo_rings
        )
        
        indices = chunk.iter_all() if include_halo else chunk.iter_core()
        for idx in indices:
            pos = chunk.index_to_hexposition(idx)
            source = grid.hexposition_to_index(pos, origin_index=origin)
            if 0 <= source < len(grid.hexes):
                region.hexes.add(source)
        return region

    def compute_spiral_rings(self) -> int:
        """Compute number of chunk rings needed to cover the grid."""
        grid = self.terrain.hexGrid
        min_dim = min(grid.nRows, grid.nCols)
        # Each chunk covers 2*rings+1 diameter, spacing is 2*rings
        # Need enough rings to reach from center to edge
        chunk_diameter = 2 * self.rings
        return max(1, int(np.ceil(min_dim / chunk_diameter / 2)))

    def spiral_chunks(self, n_rings: int = None) -> list[HexChunk]:
        """Generate chunks arranged in a spiral pattern.
        
        Args:
            n_rings: Number of rings of chunks around center. 
                     If None, computed from grid dimensions.
        """
        grid = self.terrain.hexGrid
        
        if n_rings is None:
            n_rings = self.compute_spiral_rings()
        
        # Distance between chunk centers for edge-touching (with halo overlap)
        distance = 2 * self.rings
        
        positions = HexPosition.origin().spiral(n_rings, distance=distance)
        
        chunks = []
        for pos in positions:
            origin_idx = grid.hexposition_to_index(pos, origin_index=grid.middle)
            
            if origin_idx < 0 or origin_idx >= len(grid.hexes):
                continue
                
            chunk = self.chunkAt(origin=origin_idx)
            chunk.position = pos
            chunks.append(chunk)
        
        return chunks
```

Usage would be:

```python
cover = ChunkCover(demoTerr, rings=3, halo_rings=1)

for chunk in cover.spiral_chunks():  # auto-computes n_rings
    chunk.apply_styles(demoTerr.colorLevels, elevation_delta=demoTerr.elevationDelta)
    centerPos = grid.hexes[chunk.origin].center
    chunkLayer += chunk.render_svg(grid.radius, center=centerPos, core_only=True)
```

In [ ]:
def demo_region_spiral_with_arrows(distance=1):
    """Demo showing a region growing in a spiral pattern with arrows"""
    demoTerr = TerraDemo().sanFran()
    demoTerr.colorMap()
    demoTerr.hexGrid.update()
    textStyle = StyleCSS("textStyle",fill="white",font_size="2em",filter=" drop-shadow( 3px 3px 2px rgba(0, 0, 0, .7))")
    demoTerr.hexGrid.builder.add_style(textStyle)
    demoTerr.hexGrid.builder.add_centered_text(
            "I don’t much care where– said Alice.", 
            y_offset=-demoTerr.hexGrid.builder.height/2 + 70,
            class_name=textStyle.name
        )

    grid =  demoTerr.hexGrid
    
    
    # Create gradient of colors
    colors = StyleCSS.seaborn("BuGn",37)
    
    arrow_style = StyleCSS("spiral_arrow", stroke="#333333", stroke_width=2, fill="none")
    
    for color in colors:
        grid.builder.add_style(color)
    grid.builder.add_style(arrow_style)
    
    center = grid.middle
    
    # Use the spiral method from HexPosition
    spiral_positions = HexPosition.origin().spiral(3,distance=distance)
    
    arrow_layer = ""
    prev_idx = None
    
    for i, pos in enumerate(spiral_positions):
        idx = grid.hexposition_to_index(pos, center)
        if 0 <= idx < len(grid.hexes):
            grid.hexes[idx].style = colors[i % len(colors)]
            #grid.hexes[idx].label = str(i)
            print(idx)
            
            # Draw arrow from previous hex to current
            if prev_idx is not None and prev_idx >= 0:
                arrow_layer += grid.arrow(prev_idx, idx, style=arrow_style)
            
            prev_idx = idx
    
    grid.update()
    grid.builder.adjust("arrows", arrow_layer)
    return grid.builder.show()

In [ ]:
SVGBuilder.BUILDERHIDE = False
demo_region_spiral_with_arrows(5)

465
470
618
613
460
313
318
323
475
623
770
765
760
608
455
308
160
165
170
175
328
628
775
755
603
450
303
155
8
13
18
23


so I want to make a series of chunks that go around in the spiral pattern in the ChunkCover

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice idea! So you want to tile chunks in a spiral pattern around a center chunk.

A key question: what's the relationship between chunk positions? Looking at `HexChunk.center_world`, it uses `spacing = 2 * core_rings` to convert chunk coordinates to world coordinates. That means adjacent chunks (in chunk-space) would have their centers separated by `2 * rings` hex-steps.

Do you want:
1. **Tightly packed chunks** (edges touching, no overlap)
2. **Overlapping chunks** (with halos overlapping neighboring cores)
3. **Spaced out chunks** (gaps between them)

For option 1, if your chunks have `rings=4`, you'd want chunk centers separated by about 8-9 hex positions.

Also, are you thinking of:
- Adding a method like `spiral_chunks(n_rings)` to `ChunkCover` that returns a list of chunks?
- Or iterating chunk *positions* first, then calling `chunkAt()` for each?

so when we are doing the spiral at a even distance there is a midpoint. in our spiral case of 6 each chunk can have a radius of 3. the first two rings are inside and the 3 ring is the halo that intersects. this will give us nice consisent coverage. Is there a better way without overlaping?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're on the right track! With distance=6 and chunk radius=3, the chunks would touch edge-to-edge perfectly (3+3=6) with no overlap and no gaps.

If you don't *want* overlapping, you can skip the halo entirely and just use `rings=3` with full core. The halo is mainly useful when you need smooth data blending between chunks (like terrain generation where you want to avoid visible seams).

For pure non-overlapping coverage:
- `distance = 2 * rings` gives perfect edge-touching
- No halo needed—all hexes are core

Does that match what you're aiming for? Or do you have a use case where you'd still want some overlap for blending purposes?

Great lets do the no halo needed

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Perfect! So the plan is:
- `rings = 3` (or whatever size you want)
- `distance = 2 * rings` (so distance=6 for rings=3)
- Use `HexPosition.origin().spiral(n_rings, distance=distance)` to get chunk center positions

You could add a method to `ChunkCover` like `spiral_chunks(n_rings)` that:
1. Computes the spiral positions using the distance formula
2. For each position, creates a chunk at that location
3. Returns the list of chunks

Want to take a stab at writing that method, or would you like me to sketch it out for you?

Can you write the chunkCover spiral_chunks(n_rings)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def spiral_chunks(self: ChunkCover, n_rings: int) -> list[HexChunk]:
    """Generate chunks arranged in a spiral pattern with no overlap.
    
    Args:
        n_rings: Number of rings of chunks around the center chunk
        
    Returns:
        List of HexChunks covering the spiral pattern
    """
    grid = self.terrain.hexGrid
    
    # Distance between chunk centers = 2 * chunk_rings for edge-touching
    distance = 2 * self.rings
    
    # Get spiral of chunk center positions
    positions = HexPosition.origin().spiral(n_rings, distance=distance)
    
    chunks = []
    for pos in positions:
        # Convert HexPosition to grid index using grid's middle as origin
        origin_idx = grid.hexposition_to_index(pos, origin_index=grid.middle)
        
        # Skip if this chunk center is outside the grid
        if origin_idx < 0 or origin_idx >= len(grid.hexes):
            continue
            
        chunk = self.chunkAt(origin=origin_idx)
        chunk.position = pos  # Store the chunk's position in chunk-space
        chunks.append(chunk)
    
    return chunks
```

Usage would look like:

```python
cover = ChunkCover(demoTerr, rings=3)
chunks = cover.spiral_chunks(n_rings=2)  # Center + 2 rings of chunks
```

Note: There's a bug in your original `chunkAt` method—it references `index` instead of `origin` in the `if index is None` check. You'll want to fix that:

```python
if origin is None:  # not "if index is None"
    origin = grid.middle
```

Can you write the full class

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
class ChunkCover:
    """Covers a terrain with HexChunks arranged in patterns."""

    def __init__(self, terrain, rings):
        self.terrain = terrain
        self.rings = rings

    def chunkAt(self, origin=None) -> HexChunk:
        """Create a chunk centered at the given grid index."""
        grid = self.terrain.hexGrid
        if origin is None:
            origin = grid.middle

        chunk = HexChunk(HexPosition.origin(), rings=self.rings)
        chunkSize = len(chunk.elevations)

        for prop in self.terrain.fields.keys():
            chunk.fields[prop] = np.zeros(chunkSize)

        for i in chunk.iter_core_with_pos():
            source = grid.hexposition_to_index(i.pos, origin_index=origin)
            if 0 <= source < len(self.terrain.elevations):
                chunk.elevations[i.idx] = self.terrain.elevations[source]
                for prop in self.terrain.fields.keys():
                    chunk.fields[prop][i.idx] = self.terrain.fields[prop][source]
        return chunk

    def regionAt(self, origin=None) -> HexRegion:
        """Create a HexRegion covering the chunk area."""
        grid = self.terrain.hexGrid
        if origin is None:
            origin = grid.middle

        region = HexRegion(set(), hexGrid=grid)
        chunk = HexChunk(HexPosition.origin(), rings=self.rings)
        for i in chunk.iter_core_with_pos():
            source = grid.hexposition_to_index(i.pos, origin_index=origin)
            if 0 <= source < len(grid.hexes):
                region.hexes.add(source)
        return region

    def spiral_chunks(self, n_rings: int) -> list[HexChunk]:
        """Generate chunks arranged in a spiral pattern with no overlap.
        
        Args:
            n_rings: Number of rings of chunks around the center chunk
            
        Returns:
            List of HexChunks covering the spiral pattern
        """
        grid = self.terrain.hexGrid
        
        # Distance between chunk centers = 2 * chunk_rings for edge-touching
        distance = 2 * self.rings
        
        # Get spiral of chunk center positions
        positions = HexPosition.origin().spiral(n_rings, distance=distance)
        
        chunks = []
        for pos in positions:
            # Convert HexPosition to grid index using grid's middle as origin
            origin_idx = grid.hexposition_to_index(pos, origin_index=grid.middle)
            
            # Skip if this chunk center is outside the grid
            if origin_idx < 0 or origin_idx >= len(grid.hexes):
                continue
                
            chunk = self.chunkAt(origin=origin_idx)
            chunk.position = pos  # Store the chunk's position in chunk-space
            chunks.append(chunk)
        
        return chunks
```

I also added bounds checking in `chunkAt` and `regionAt` to handle cases where the chunk extends beyond the terrain edges.